# 04 - Inference, control and CPU benchmarking

Everything here runs on CPU, which is the deployment target. Use this notebook
to measure real-time factor, to inspect and correct pronunciations, and to
export a deployable bundle.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp0_small.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

In [ ]:
import torch
torch.set_num_threads(os.cpu_count() or 4)
from adaptts.infer.pipeline import AdapTTS, save_wav

tts = AdapTTS.from_checkpoints(CONFIG, device="cpu")
print("loaded on CPU with", torch.get_num_threads(), "threads")

## 1. Inspect: what does the model think each word means?

`analyze` returns a plan. It lists every ambiguous word, the reading chosen, the
confidence, and the difficulty that drives adaptive depth.

In [ ]:
plan = tts.analyze(
    "انا كنت مصر على ان مصر عندها امكانيات و موارد تخليها تتفوق على دول من اللي شايفين نفسهم دول"
)
print(plan)

## 2. Correct: override a reading without diacritics

The pronunciation decision is a discrete code, so changing it is a legal edit
that the acoustic model was trained to consume.

In [ ]:
from IPython.display import Audio, display

plan = tts.analyze("انا شوفت علم مصر بيرفرف")
wav, st = tts.synthesize(plan)
print("model's own choice, code", plan.hard_words[0].code)
display(Audio(wav, rate=st["sample_rate"]))

plan.set_code("علم", 1)      # force the other reading
wav, st = tts.synthesize(plan)
print("forced to code 1")
display(Audio(wav, rate=st["sample_rate"]))

In [ ]:
# When a word appears twice with different readings, target one occurrence.
plan = tts.analyze("انا كنت مصر على ان مصر عندها امكانيات")
print(plan)
plan.set_code("مصر", 0, occurrence=0)   # first مصر only
plan.set_code("مصر", 1, occurrence=1)   # second مصر only
wav, st = tts.synthesize(plan)
display(Audio(wav, rate=st["sample_rate"]))

## 3. Benchmark on CPU

Real-time factor below 1.0 means faster than real time. Pocket TTS reports about
6x real time on an M4; expect a similar order here at the shallow exits.

In [ ]:
import time
import numpy as np

sentences = [
    "الجو النهارده حلو جدا",
    "احنا رايحين السوق بكرة الصبح ان شاء الله",
    "انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول",
]

rows = []
for txt in sentences:
    plan = tts.analyze(txt)
    tts.synthesize(plan)                       # warm up
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        wav, st = tts.synthesize(plan)
        times.append(time.perf_counter() - t0)
    rows.append((len(txt), plan.sentence_difficulty, st["depth"],
                 st["audio_seconds"], float(np.median(times))))

print(f"{'chars':>6}{'difficulty':>12}{'depth':>7}{'audio_s':>9}{'wall_s':>8}{'RTF':>7}")
print("-" * 49)
for n, d, dep, a, t in rows:
    print(f"{n:>6}{d:>12.3f}{dep:>7}{a:>9.2f}{t:>8.2f}{t / a:>7.2f}")

In [ ]:
# Adaptive versus fixed depth, on the same sentences.
import numpy as np

def bench(txt, depth=None):
    plan = tts.analyze(txt)
    if depth:
        plan.set_depth(depth)
    tts.synthesize(plan)
    ts = []
    for _ in range(3):
        t0 = time.perf_counter()
        _, st = tts.synthesize(plan)
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)), st["depth"]

full = cfg.acoustic.exit_layers[-1]
print(f"{'sentence':<34}{'adaptive':>12}{'fixed-full':>12}{'saving':>9}")
print("-" * 67)
for txt in sentences:
    ta, da = bench(txt)
    tf, _ = bench(txt, full)
    print(f"{txt[:32]:<34}{ta:>10.2f}s{tf:>10.2f}s{(1 - ta / tf) * 100:>8.0f}%")

## 4. Save audio to disk

In [ ]:
wav, st = tts.tts("اهلا بيكم في التجربة الاولى من النظام الجديد")
save_wav("runs/exp1/samples/demo.wav", wav, st["sample_rate"])
print("wrote runs/exp1/samples/demo.wav", st)
display(Audio(wav, rate=st["sample_rate"]))

## 5. Export a deployment bundle

Collects the two checkpoints, the vocabulary, the discovered codes and the
config into one directory that can be copied to a device.

In [ ]:
import shutil, json
from pathlib import Path

out = Path("runs/exp1/deploy")
out.mkdir(parents=True, exist_ok=True)
for src, dst in [
    (Path(cfg.paths.ckpt_dir) / "context_encoder" / "best.pt", "context_encoder.pt"),
    (Path(cfg.paths.ckpt_dir) / "acoustic" / "best.pt", "acoustic.pt"),
    (Path(cfg.paths.charvocab_path), "char_vocab.json"),
    # The reading lexicon, not the retired clustering one. Inference needs it to
    # know how many readings a word has and what each code means.
    (Path(cfg.paths.reading_lexicon_path), "reading_lexicon.json"),
    (Path("configs/exp1_egyptian.yaml"), "config.yaml"),
]:
    if Path(src).exists():
        shutil.copy(src, out / dst)
        print("copied", dst, f"{Path(src).stat().st_size / 1e6:.1f} MB")
    else:
        print("MISSING", src)
print()
print("bundle at", out.resolve())